# Full conditional score ZQE — Poisson (appendix probe)

**Does the *full conditional score* beat the paper's *data-anchored* term — and does
posterior sampling unlock it?** Short answer from this notebook: **no**, and the
convergence diagnostics show it's the estimator, not the optimiser.

- **Data-anchored** (the paper): $\psi=\mathbb E_q[D^\top T(y)]$ — code `g.zq_log` ($T\!\cdot\!\eta$).
- **Full score**: $\psi_{\text{full}}=\mathbb E_q[D^\top(T(y)-\mu(z))]=\mathbb E_q[\nabla_\theta\log p(y\mid z)]$ — code `g.log_prob`.
  With the *exact* posterior this is the marginal score ($\mathbb E_\theta[\psi]=0$) → the **MLE**.

We compare four encoders for the centered loss $-(m_1-m_2)$, at the stress cell
$n=50$, with two convergence diagnostics: **tail-drift** (Polyak half-vs-half, ≈0 if
settled) and **grad-norm** $\|\overline{\nabla W}\|/\|W\|$ (≈0 at a root).

## Definitive result (3 seeds × 3000 steps, $q=2,p=20,n=50$)

| method | Procrustes | scale ‖Ŵ‖/‖W‖ | tail-drift | grad-norm | converged? |
|---|---|---|---|---|---|
| anchored + gauss-MAP (paper) | **0.242** | 0.97 | 0.012 | 0.000 | ✓ |
| full + gauss-MAP (log1p) | 0.717 | 0.30 | 0.055 | 7.66 | ✗ not converged |
| full + **Poisson-MAP** (proper point) | 0.314 | 1.02 | 0.019 | 0.002 | ✓ |
| full + Poisson-**Laplace** (sampled) | 0.787 | 1.51 | 0.150 | 6.5e5 | ✗ diverged |

**Reading.**
1. `full+log1p`'s apparent shrinkage was **non-convergence** (grad-norm 7.66) from a
   misspecified encoder — a computation artifact, not the estimator.
2. With a **proper Poisson MAP** encoder the full score **converges cleanly** (grad
   0.002, scale 1.02 — no shrinkage) but still **loses to data-anchored** (0.314 vs
   0.242). The data-anchored term is genuinely more accurate at small $n$, not just
   more robust to bad encoders.
3. **Posterior sampling (Laplace) is numerically unstable** here (grad-norm 6e5):
   near-singular per-observation Hessians give extreme $z$ → $e^\eta$ blow-ups. A
   usable sampler would need real variance control — future work, not a quick win.

**Conclusion: keep the data-anchored MAP design.** It is the most accurate *and* the
most stable, even against a properly-encoded full score. (`n=500`: anchored 0.077,
full+log1p 0.109, full+gauss-sampled 0.114 — full score never wins.)

The cell below re-runs a lighter version (2 seeds × 2000 steps) so the notebook is
self-contained; expect the same qualitative picture.

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np, torch, torch.nn as nn
from gllvm.gllvm_module import GLLVM
from gllvm.glms import PoissonGLM
from gllvm.encoder import MapEncoderGaussianLog1p, MapEncoderPoissonNewton
from gllvm.autofit import procrustes_error
from gllvm.simulations import make_mixed, simulate
DEV = "cuda" if torch.cuda.is_available() else "cpu"
SCALE = 1.0; q, p = 2, 20
print("device:", DEV)

def fresh():
    g = GLLVM(latent_dim=q, output_dim=p, bias=True).to(DEV)
    g.add_glm(PoissonGLM, idx=list(range(p)), params={"T": torch.log1p}, name="P")
    with torch.no_grad():
        nn.init.normal_(g.wz, std=SCALE); nn.init.zeros_(g.bias)
    return g

class LaplacePoisson:
    """Proper Poisson posterior: Newton-MAP mean + true Poisson Hessian curvature, sampled."""
    def __init__(self, g, lam=1.0): self.g = g; self.map = MapEncoderPoissonNewton(g, lam=lam, max_iter=30)
    def sample(self, y):
        W = self.g.wz.detach(); b = self.g.bias.detach()
        zmap = self.map.forward(y)
        eta = (zmap @ W.T + b).clamp(max=10); mu = torch.exp(eta)
        H = torch.einsum("pa,np,pb->nab", W, mu, W) + torch.eye(q, device=W.device)
        L = torch.linalg.cholesky(H)
        eps = torch.randn(zmap.shape[0], q, 1, device=W.device)
        y_ = torch.linalg.solve_triangular(L.transpose(-1, -2), eps, upper=True).squeeze(-1)
        return zmap + y_, zmap, None

ENC = {"gauss-map": lambda g: MapEncoderGaussianLog1p(g),
       "pois-map":  lambda g: MapEncoderPoissonNewton(g, lam=1.0, max_iter=30),
       "pois-laplace": lambda g: LaplacePoisson(g)}

def fit(Y, W, mode, enc_kind, npost, steps=2000, lr=0.03, seed=0, tail=0.5, clip=5.0):
    torch.manual_seed(seed); g = fresh(); enc = ENC[enc_kind](g)
    opt = torch.optim.Adam(g.parameters(), lr=lr); n = len(Y); start = int((1 - tail) * steps)
    sc = (lambda y, z: g.log_prob(y, z=z)) if mode == "full" else (lambda y, z: g.zq_log(y, z=z))
    a1 = torch.zeros_like(g.wz); a2 = torch.zeros_like(g.wz); c1 = c2 = 0
    gbar = torch.zeros_like(g.wz); gc = 0; mid = start + (steps - start) // 2
    for t in range(steps):
        with torch.no_grad():
            yq = g.sample(z=g.sample_z(n))
            zs  = [enc.sample(Y)[0]  for _ in range(npost)]
            zqs = [enc.sample(yq)[0] for _ in range(npost)]
        m1 = torch.stack([sc(Y,  z ).sum(-1).mean() for z  in zs ]).mean()
        m2 = torch.stack([sc(yq, zq).sum(-1).mean() for zq in zqs]).mean()
        loss = -(m1 - m2); opt.zero_grad(); loss.backward()
        graw = g.wz.grad.detach().clone()
        torch.nn.utils.clip_grad_norm_(g.parameters(), clip); opt.step()
        if t >= start:
            (a1 if t < mid else a2).add_(g.wz.detach());
            if t < mid: c1 += 1
            else: c2 += 1
            gbar += graw; gc += 1
    wz = (a1 + a2) / (c1 + c2)
    drift = procrustes_error(a1 / c1, a2 / c2)
    gnorm = float((gbar / gc).norm() / wz.norm())
    return procrustes_error(W, wz), float(wz.norm() / torch.as_tensor(np.asarray(W)).norm()), drift, gnorm

In [ ]:
cfg = [("anchored+gauss-map",      "anchored", "gauss-map",    1),
       ("full+gauss-map(log1p)",    "full",     "gauss-map",    1),
       ("full+pois-MAP(point)",     "full",     "pois-map",     1),
       ("full+pois-LAPLACE(samp)",  "full",     "pois-laplace", 8)]
print(f"n=50  (q={q}, p={p})   [converged if drift & gradN ~ 0]")
print(f"{'method':<26}{'procr':<8}{'scale':<7}{'drift':<8}{'gradN':<10} seeds")
for lab, mode, enc, npost in cfg:
    R = []
    for seed in range(2):
        torch.manual_seed(seed); gt = make_mixed(n_latent=q, poisson=p, wz_scale=SCALE)
        Y, _ = simulate(gt, n_samples=50, device="cpu"); W = gt.wz.detach()
        R.append(fit(Y.to(DEV), W, mode, enc, npost, seed=seed))
    pe, sc_, dr, gn = (np.mean([r[i] for r in R]) for i in range(4))
    print(f"{lab:<26}{pe:<8.3f}{sc_:<7.2f}{dr:<8.3f}{gn:<10.3g} {[round(r[0],2) for r in R]}")

## Takeaway

The data-anchored MAP estimator wins on accuracy **and** stability, even versus a
properly-encoded full score, and the full-score sampler doesn't even converge here.
This is the appendix that justifies the paper's design choice — and parks
"efficient + robust ZQE via a *stable* sampled posterior" as honest future work.